In [0]:
bronze_path = f"/Volumes/lakehouse_portfolio/bronze/breweries_raw/year=2026/month=03/day=27/"

#breweries_df = spark.read.option.("multiline",  "true")json(bronze_path)
breweries_df = spark.read.option("multiLine", "true").json(bronze_path)


print(f"Total de Registros lidos: {breweries_df.count()}")
breweries_df.printSchema()
breweries_df.show(5, truncate=False)



In [0]:
from pyspark.sql import functions as F
breweries_df.select([
                     F.count(F.when(F.col(c).isNull(), c)).alias(c)
                     for c in breweries_df.columns
                    ]).show()

In [0]:
silver_df = (
    breweries_df
    .drop("address_2", "address_3", "street")
    .filter(F.col("name").isNotNull())
    .filter(F.col("state").isNotNull())
    .withColumn("name", F.trim(F.col("name")))
    .withColumn("city", F.trim(F.col("city")))
    .withColumn("state", F.trim(F.col("state")))
    .withColumn("brewery_type", F.lower(F.col("brewery_type")))
)
print(f"Bronze: {breweries_df.count()} registros")
print(f"Silver: {silver_df.count()} registros")
silver_df.show(5, truncate=False)

In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("state") \
    .saveAsTable("lakehouse_portfolio.silver.breweries")
print("Tabela Silver salva com sucesso!")
print("Local: lakehouse_portfolio.silver.breweries")

In [0]:
# Lê a tabela que acabou de salvar e valida
tabela_silver = spark.table("lakehouse_portfolio.silver.breweries")

print(f"✅ Total de registros na Silver: {tabela_silver.count()}")
print(f"📊 Estados distintos: {tabela_silver.select('state').distinct().count()}")
print(f"🍺 Tipos de cervejaria: {tabela_silver.select('brewery_type').distinct().count()}")

# Distribuição por tipo
tabela_silver.groupBy("brewery_type") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()